# 16 — The platform landscape, with Foundry as the worked example

You have now built the loop, given it tools, moved those tools to another process, wrapped the same desk in two frameworks, retrieved a corpus, guarded a ticket, and printed a bill.

A vendor is not selling you "an AI agent". They are selling one or more of **six layers** you have already written by hand. This chapter names the layers, says which ones are open standards, and says where lock-in actually lives.

You will run the Chinook desk one more time in this notebook — on the **same API surface a managed runtime exposes** — and then put Microsoft Foundry's vocabulary next to it, field by field. Microsoft Foundry is the one managed path in this course, because Azure is the one cloud. **A naming note before you search for any of this:** the product was Azure AI Foundry and is now **Microsoft Foundry**. The older Agent Service surface is documented as *Foundry (classic)*. You will meet all three names in Microsoft's own docs; they are the same product line. AWS and Google sell the same six layers under different names. One sentence of orientation, then we stay here. No student needs an Azure credential. If the instructor did not provision a project, the captured shape is the lesson.


## 1. Learn

```
08  you wrote the desk
09  you drew the arrows
15  you scored one run
16  you are here — who sells each layer, and what you give up
17  whether any of it should exist
```

**Your notebook vs a managed runtime**

```mermaid
flowchart LR
    subgraph Yours["your notebook"]
        A["question"] --> B["your for-loop"]
        B --> C["your tools"]
    end
    subgraph Theirs["Foundry Agent Service"]
        D["question"] --> E["their runtime"]
        E --> F["their tool catalog"]
    end
```

The picture on the left is modules 03 to 15. The picture on the right is the same loop, as a product. The question for a buying meeting is not "should we use agents?" It is "which of these six layers are we buying, and which do we already have?"

| Layer | Problem it solves | The open pattern | What a cloud sells you | You already built |
|---|---|---|---|---|
| Model | Reasoning and tool selection | Any compatible endpoint | A managed model catalogue | Every `create` since 00 |
| Tools | Hands, portably | **MCP** | A governed tool catalogue | 02, then 06 |
| Orchestration | Control flow, state, retries | Open frameworks | A managed agent runtime | 03, then 08, then 09 |
| Context | Memory and grounding | Open vector stores | A managed search or memory service | the `messages` list; Chroma in 11 |
| Delegation | Agents talking to agents | **A2A** | A connected-agents feature | 13 |
| Governance | Identity, safety, evals, cost | OpenTelemetry, open evals | Identity, filters, hosted traces | 14 and 15 |

The two bold rows are open standards. Everything in the "cloud sells you" column is a vendor's implementation of a pattern you can build yourself. That is why the course built the pattern first.

AWS sells the same six as Bedrock / AgentCore. Google sells them as Vertex / Agent Engine. The names move. The layers do not.

Vendors also sort by **verb**, not by logo:

| Who | Verb | What you are buying |
|---|---|---|
| Non-technical | **use** | A finished application. Someone else chose the model, the tools, and the loop. |
| Non-technical | **build** | A visual studio. You configure an agent without writing the loop — and inherit whatever the vendor did not anticipate. |
| Technical | **develop** | The loop is yours. That is where these two days live. |
| Technical | **execute** | Managed infrastructure that runs a loop you wrote. This is where portability stops. |

These two days are **develop**. Foundry Agent Service is **execute**, with a **build** surface (Agent Builder) next to it. A graph framework you can also buy hosted sits in two columns at once. That is the business model, not sloppiness.

**Where lock-in lives**

```mermaid
flowchart TD
    P["portable"] --> M["model"]
    P --> T["tools behind MCP"]
    P --> O["open orchestration"]
    S["sticky"] --> Mem["memory"]
    S --> I["identity"]
    S --> Obs["observability"]
```

- **Portable:** the model (a compatible interface and a re-run of your evals). Tools, if they sit behind MCP. Orchestration, if you stayed on an open framework.
- **Sticky:** memory, identity, and the audit trail. Migrating a vector store is annoying. Migrating an identity model and a year of traces is a project.

Nobody argues about the model any more. The arguments start about eighteen months in, in the sticky column.

**What we will not do.** The classic Foundry surface was threads, runs, and messages. It is deprecated (retirement 31 March 2027). The current surface is **conversations** and **responses**, with a prompt-agent definition (`model`, `instructions`, `tools`). We do not install an Azure SDK and we do not call either API from this notebook. The lesson is the mapping, not a second client.

Cut first if the room is behind: the optional portal demo. Keep the six-layer table and the lock-in split.


## 2. Do

### Load the environment

Same load as every other module. The OpenAI key is unused here. The two Foundry names are optional. Neither value is printed.


In [1]:
from pathlib import Path
import os
from urllib.parse import urlparse

from dotenv import load_dotenv, find_dotenv


load_dotenv(find_dotenv(usecwd=True))
ROOT = Path(find_dotenv(usecwd=True)).parent

api_key = os.environ.get("OPENAI_API_KEY", "").strip()
foundry_endpoint = os.environ.get("AZURE_FOUNDRY_PROJECT_ENDPOINT", "").strip()
foundry_key = os.environ.get("AZURE_FOUNDRY_API_KEY", "").strip()

print("OPENAI_API_KEY is set:", bool(api_key))
print("AZURE_FOUNDRY_PROJECT_ENDPOINT is set:", bool(foundry_endpoint))
print("AZURE_FOUNDRY_API_KEY is set:", bool(foundry_key))
print("ROOT:", ROOT)
if foundry_endpoint:
    print("Foundry host:", urlparse(foundry_endpoint).netloc)


OPENAI_API_KEY is set: True
AZURE_FOUNDRY_PROJECT_ENDPOINT is set: False
AZURE_FOUNDRY_API_KEY is set: False
ROOT: /Users/tarekatwan/Downloads/ai_agents_course


### The same desk, on the surface a managed runtime exposes

Every module since 00 used **Chat Completions**: you own the `messages` list, you send the whole thing every turn.

A managed runtime does not work that way. It keeps the conversation server-side and hands you an id. Microsoft Foundry's current surface is **`conversations` + `responses`**, and OpenAI exposes exactly the same two objects on the key you already have. So we can run the real shape here, with no Azure account, and then read Foundry's version of it.

Two differences from every earlier module, and they are the whole point:

- **The conversation has an id.** `conv_...` lives on the server. You stop carrying the transcript.
- **Tool results go back by `call_id`**, as `function_call_output` items — not as a `tool` message you append to a list you own.

The loop underneath is still module 03's `for`. Same three tools, same Helena question, same three facts.

In [ ]:
import json
import sqlite3
import unicodedata

from openai import OpenAI

client = OpenAI()
model = os.environ["MODEL_DEFAULT"]

db = sqlite3.connect(ROOT / "data" / "chinook.db", check_same_thread=False)
db.row_factory = sqlite3.Row


def fold(text: str) -> str:
    nfkd = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in nfkd if not unicodedata.combining(ch)).casefold()


def _row(name: str):
    needle = fold(name).strip()
    for r in db.execute("select CustomerId, FirstName, LastName, SupportRepId from customers"):
        if needle in fold(r["FirstName"] + " " + r["LastName"]):
            return r


def invoice_count(name: str) -> str:
    r = _row(name)
    n = db.execute("select count(*) from invoices where CustomerId = ?", (r["CustomerId"],)).fetchone()[0]
    return f"{r['FirstName']} {r['LastName']} has {n} invoices"


def invoice_total(name: str) -> str:
    r = _row(name)
    t = db.execute("select round(sum(Total), 2) from invoices where CustomerId = ?", (r["CustomerId"],)).fetchone()[0]
    return f"{r['FirstName']} {r['LastName']} has spent {t} dollars"


def support_rep(name: str) -> str:
    r = _row(name)
    e = db.execute("select FirstName || ' ' || LastName from employees where EmployeeId = ?",
                   (r["SupportRepId"],)).fetchone()[0]
    return f"{r['FirstName']} {r['LastName']}'s support rep is {e}"


RUN = {"invoice_count": invoice_count, "invoice_total": invoice_total, "support_rep": support_rep}


def schema(name, description):
    return {
        "type": "function",
        "name": name,
        "description": description,
        "parameters": {
            "type": "object",
            "properties": {"name": {"type": "string"}},
            "required": ["name"],
            "additionalProperties": False,
        },
    }


TOOLS = [
    schema("invoice_count", "How many invoices a customer has."),
    schema("invoice_total", "What a customer has spent."),
    schema("support_rep", "Who the customer's support representative is."),
]

INSTRUCTIONS = "Use the tools. Do not invent numbers or names."

# No model yet. Same discipline as 08, 09 and 15.
print(invoice_count("Helena"))
print(invoice_total("Helena Holy"))
print(support_rep("Helena Holy"))

7, 49.62, Steve Johnson. Cover those before any model runs.

Now the loop. Create a conversation, send the question, run whatever it asks for, send the results back by `call_id`. Cap the turns, exactly as you have since module 03.

In [ ]:
QUESTION = (
    "How many invoices does Helena Holy have, what is her total spend, "
    "and who is her support representative?"
)

conversation = client.conversations.create()
print("conversation:", conversation.id)

payload = [{"role": "user", "content": QUESTION}]
n_tools = 0
final_text = ""

for turn in range(5):
    response = client.responses.create(
        model=model,
        conversation=conversation.id,
        instructions=INSTRUCTIONS,
        input=payload,
        tools=TOOLS,
        reasoning={"effort": "none"},
    )
    calls = [o for o in response.output if o.type == "function_call"]
    print(f"--- turn {turn + 1}: {[o.type for o in response.output]}")

    if not calls:
        final_text = response.output_text
        break

    payload = []
    for call in calls:
        args = json.loads(call.arguments)
        result = RUN[call.name](args["name"])
        n_tools += 1
        print(f"    {call.name}({args['name']!r}) -> {result}")
        payload.append({"type": "function_call_output", "call_id": call.call_id, "output": result})

print()
print(final_text)
print()
print("n_tools:", n_tools, "| tokens:", response.usage.input_tokens, "in /", response.usage.output_tokens, "out")

You never held the transcript. The server did, under that `conv_` id — and you still decided every tool call, exactly as in module 02.

### Now Foundry's vocabulary for the same thing

Below is that same desk written the way Microsoft Foundry stores a **prompt agent**: a name, a model deployment, instructions, and a tool list. The field names match the current `PromptAgentDefinition`.

Read it next to what you just ran. It is not a new idea — it is the same four things, plus somebody else's `for` loop.

In [ ]:
CAPTURED_AGENT = {
    "name": "chinook-desk",
    "kind": "prompt",
    "model": "<a Foundry model deployment name>",
    "instructions": INSTRUCTIONS,
    "tools": [{"name": t["name"], "description": t["description"]} for t in TOOLS],
}

print("Foundry prompt agent:")
for k, v in CAPTURED_AGENT.items():
    print(f"  {k:14} {v if not isinstance(v, list) else [t['name'] for t in v]}")

print()
print(f"{'Foundry field':26} {'what you just ran':38} where you first built it")
print("-" * 104)
mapping = [
    ("name", "the 'chinook-desk' label", "08 called it desk_tools"),
    ("model", model, "MODEL_DEFAULT, since module 00"),
    ("instructions", INSTRUCTIONS[:36], "the system message, module 00"),
    ("tools", f"{len(TOOLS)} named functions", "the JSON you typed in module 02"),
    ("conversation", conversation.id[:34] + "...", "the messages list you owned until now"),
    ("responses", f"{n_tools} tool calls, {turn + 1} turns", "the for-loop from module 03"),
    ("output_text", final_text.replace(chr(10), " ")[:36] + "...", "the last assistant sentence"),
]
for field, ours, built in mapping:
    print(f"{field:26} {str(ours):38} {built}")

print()
print("The managed product stores the first four and runs the last three for you.")
print("That is the trade. Nothing in this table is a new idea.")

### The Azure half — an instructor demo, not a student lab

Everything above ran on the class key. Nothing above needed Azure. That is deliberate.

**No student creates an Azure resource in this course, and nobody is handed a tenant key.** Two reasons, and the second is the one that matters:

- A shared model deployment has TPM/RPM quota. Twenty people running one cell together get HTTP 429, and the module becomes a debugging session.
- A Foundry API key is **all-or-nothing** on that resource. Handing one to twenty people, in a course whose security module is about credentials reaching places they should not, would be a poor lesson to end on. Module 16's own Observe table says identity is the sticky layer; slide 93 says shared service accounts are the anti-pattern.

So the instructor drives, on a project they provisioned and will delete afterwards. Four things, then sit down:

1. **Agent Builder** — the same four fields you just printed, as a form.
2. **Connected Agents** — module 13's desk, as a product feature.
3. **The trace view** — module 15's ledger and message list, as a timeline.
4. **Content filters / prompt shields** — a hosted version of the Python scan from module 14. Still incomplete. Still worth having.

Watch for one thing in the trace view: **it stores your prompts and your tool arguments.** That is customer data on someone else's timeline. Decide what you redact before you switch it on, not after.

If the two Foundry names in `.env` are empty, the next cell says so and the module is complete without them. The live run above was the lab.

In [4]:
if foundry_endpoint:
    print("Foundry endpoint set. The live walk is the portal, not this cell.")
    print("Host only:", urlparse(foundry_endpoint).netloc)
    print("Instructor: Agent Builder -> Connected Agents -> trace view -> content filters. Then delete the resource group.")
else:
    print("No Foundry endpoint set, and nothing here needs one.")
    print("The lab above ran on the class key, on the same conversations + responses surface Foundry exposes.")

No Foundry endpoint. Designed path: the captured shape is the lesson.


## 3. Observe

Same Chinook question. Same three facts. What you can no longer see, once the runtime is theirs.


In [5]:
hidden = [
    ("the for-loop", "orchestration", "Their runtime. You can still cap it if they expose a turn limit."),
    ("the tool dispatch", "tools", "Portable if the tools are MCP. Sticky if they only exist in the catalogue."),
    ("the messages list", "context / traces", "You get a timeline instead of a print. Redact before you turn it on."),
    ("who the agent runs as", "identity", "Entra Agent ID, not the class OpenAI key. This is the sticky one."),
    ("the dollar ledger", "governance", "Module 15 printed it. A platform will too, on their invoice cycle."),
]
print(f"{'what you printed by hand':28} {'layer':18} what a platform does with it")
print("-" * 100)
for what, layer, note in hidden:
    print(f"{what:28} {layer:18} {note}")

print()
print("portable:", "model, MCP tools, an open graph")
print("sticky:  ", "memory, identity, the audit trail")


what you printed by hand     layer              what a platform does with it
----------------------------------------------------------------------------------------------------
the for-loop                 orchestration      Their runtime. You can still cap it if they expose a turn limit.
the tool dispatch            tools              Portable if the tools are MCP. Sticky if they only exist in the catalogue.
the messages list            context / traces   You get a timeline instead of a print. Redact before you turn it on.
who the agent runs as        identity           Entra Agent ID, not the class OpenAI key. This is the sticky one.
the dollar ledger            governance         Module 15 printed it. A platform will too, on their invoice cycle.

portable: model, MCP tools, an open graph
sticky:   memory, identity, the audit trail


## 4. Challenge

Two short classifications. The words mean the same thing they did in Learn.

**Layers.** For each row, bind `"portable"` or `"sticky"`.

**Buying.** For each job, bind `"local"` (you keep the loop you wrote), `"foundry"` (you are buying execute / identity / filters), or `"neither"` (it should not be an agent).

A couple of rows have more than one defensible answer. The assert only checks the ones that do not.

| key | What you are looking at |
|---|---|
| `layer["model"]` | Swap the chat model next quarter and re-run evals. |
| `layer["mcp_tools"]` | `get_fact` / `get_flight` behind the server from module 06. |
| `layer["open_graph"]` | The LangGraph loop from module 09, still in this repo. |
| `layer["vector_memory"]` | A year of ticket embeddings in a vendor store, with their filters. |
| `layer["identity"]` | The agent must run as itself in Entra, not on the class key. |
| `layer["hosted_traces"]` | Eighteen months of prompts and tool args in a vendor timeline. |
| `buy["nightly_csv"]` | Export yesterday's invoices to a file share, same steps every night. |
| `buy["helena_desk"]` | The three Chinook lookups you already run in 08, 09 and 15. |
| `buy["refunds_with_identity"]` | Customer-facing refunds. Risk wants an agent identity and a content filter. |


In [ ]:
layer = {}
buy = {}

# layer["model"] = ...
# layer["mcp_tools"] = ...
# layer["open_graph"] = ...
# layer["vector_memory"] = ...
# layer["identity"] = ...
# layer["hosted_traces"] = ...

# buy["nightly_csv"] = ...
# buy["helena_desk"] = ...
# buy["refunds_with_identity"] = ...

In [ ]:
allowed_layer = {"portable", "sticky"}
allowed_buy = {"local", "foundry", "neither"}

assert set(layer) == {
    "model", "mcp_tools", "open_graph",
    "vector_memory", "identity", "hosted_traces",
}, "bind every layer key"
assert set(buy) == {"nightly_csv", "helena_desk", "refunds_with_identity"}, "bind every buy key"
assert set(layer.values()) <= allowed_layer, "each layer value is portable or sticky"
assert set(buy.values()) <= allowed_buy, "each buy value is local, foundry, or neither"

assert layer["model"] == "portable", "a compatible model is a config change"
assert layer["mcp_tools"] == "portable", "MCP is the portability you already built"
assert layer["open_graph"] == "portable", "an open graph still runs here"
assert layer["identity"] == "sticky", "identity is where lock-in lives"
assert layer["hosted_traces"] == "sticky", "the audit trail is a project to move"
assert buy["nightly_csv"] == "neither", "known steps, every night, is a workflow"

print("layer:")
for k, v in layer.items():
    print(f"  {k:18} {v}")
print("buy:")
for k, v in buy.items():
    print(f"  {k:24} {v}")
